In [ ]:
import pandas as pd
import yfinance as yf

In [ ]:
PERIOD = "10y"

In [ ]:
# Get QQQ tickers from csv

csv_path = './qqq_holdings.csv'
df = pd.read_csv(csv_path)
tickers = df["Ticker"].str.strip().tolist()
# print(len(tickers), tickers[:10])

In [ ]:
# get historical price and volume data for tickers in QQQ
raw = yf.download(tickers=tickers, period=PERIOD, auto_adjust=True)
prices = raw['Close']
volumes = raw['Volume']
ticker_data = prices

In [ ]:
ticker_data

In [ ]:
def calculate_52_wk_trend(prices, n=252, lag=20):
    # calculate OLS slope using 52 week rolling window
    j = pd.Series(range(len(prices)), index=prices.index, dtype="float64")
    sum_y = prices.rolling(n).sum()
    sum_jy = prices.mul(j, axis=0).rolling(n).sum()
    mean_x = (n - 1) / 2
    var_x = n * (n ** 2 - 1) / 12
    s_xy = sum_jy - sum_y.mul(j - n + 1, axis=0)
    slope = (s_xy - mean_x * sum_y) / var_x
    return slope.shift(lag)


def pct_above_260d_low(prices, n=260, lag=20):
    return (prices / prices.rolling(n).min() - 1).shift(lag)


def wk_price_oscillator(prices, short=20, long=252, lag=20):
    ma_short = prices.rolling(short).mean()
    ma_long = prices.rolling(long).mean()
    return (ma_short / ma_long - 1).shift(lag)


def wk_39_return(prices, n=195, lag=20):
    return prices.pct_change(n).shift(lag)


def wk_51_volume_price_trend(prices, volumes, n=255, lag=20):
    # cumulative sum of volume * daily return
    # the 51-week factor is the change in VPT over the trailing n trading days.
    vpt = (volumes * prices.pct_change()).cumsum()
    return vpt.diff(n).shift(lag)


In [ ]:
def cross_sectional_zscore(df):
    # Row-wise z-score: subtract the cross-sectional mean and divide by the
    # cross-sectional std.
    return df.sub(df.mean(axis=1), axis=0).div(df.std(axis=1, ddof=0), axis=0)

# BME = business month end
def monthly_z_score(prices, volumes, freq="BME"):
    factors = {
        "trend_52w":     calculate_52_wk_trend(prices),
        "pct_above_low": pct_above_260d_low(prices),
        "oscillator":    wk_price_oscillator(prices),
        "ret_39w":       wk_39_return(prices),
        "vpt_51w":       wk_51_volume_price_trend(prices, volumes),
    }
    monthly = {name: f.resample(freq).last() for name, f in factors.items()}
    z_per_factor = {name: cross_sectional_zscore(m) for name, m in monthly.items()}
    # Stack the five z-score frames on a (factor, date) MultiIndex
    stacked = pd.concat(z_per_factor, names=["factor", "date"])
    composite = stacked.groupby(level="date").mean()
    return composite, z_per_factor


monthly_z, factor_z = monthly_z_score(prices, volumes)
monthly_z.tail(10)

In [ ]:
def select_baskets(monthly_z, n_long=10, n_short=10):
    # Pick top-N longs / bottom-N shorts and weight every pick by the
    # magnitude of its z-score. Weights are normalized across
    # both legs together so each leg's share of the portfolio is
    # proportional to the relative magnitude of z-scores.
    valid_count = monthly_z.count(axis=1)
    z = monthly_z.loc[valid_count >= n_long + n_short]

    long_rank = z.rank(axis=1, ascending=False, method="first")
    short_rank = z.rank(axis=1, ascending=True, method="first")
    long_mask = long_rank <= n_long
    short_mask = short_rank <= n_short

    # clip protects against
    # the rare case of a top-N long with z<=0 or a bottom-N short with z>=0.
    long_w = z.where(long_mask).clip(lower=0).fillna(0)
    short_w = (-z).where(short_mask).clip(lower=0).fillna(0)

    raw = long_w - short_w  # signed conviction weights, not yet normalized

    # Single normalizer across the whole row.
    # The result has |weights|.sum(axis=1) == 1 per row
    # each name's weight is its share of total conviction.
    total = raw.abs().sum(axis=1).replace(0, pd.NA)
    return raw.div(total, axis=0).fillna(0)


def baskets_to_lists(signals):
    # Variable-length list per row; .apply iterates over ~100 month rows,
    # not over the underlying data, so it is not on the hot path.
    longs = signals.where(signals > 0).apply(
        lambda r: r.dropna().index.tolist(), axis=1
    )
    shorts = signals.where(signals < 0).apply(
        lambda r: r.dropna().index.tolist(), axis=1
    )
    return pd.DataFrame({"long": longs, "short": shorts})


signals = select_baskets(monthly_z, n_long=20, n_short=5)
baskets = baskets_to_lists(signals)
baskets.tail(60)
baskets['long']['2026-04-30']

In [ ]:
qqq = yf.download("QQQ", period=PERIOD, auto_adjust=True)["Close"].squeeze()
qqq.tail()

In [ ]:
class MomentumBacktest:
    """
    Monthly-rebalanced long/short backtest.

    - `prices` is the daily wide DataFrame (Date x Ticker).
    - `signals` is a wide DataFrame at month-end with values in {-1, 0, +1}
      coming from `select_baskets`.
    - `benchmark` is a daily Series of the comparator ETF price.
    - `years` clips the report window to the trailing N years; underlying
      monthly returns are computed on the full price history so the first
      month inside the window has a valid prior price to diff against.
    """

    def __init__(self, prices, signals, benchmark, years=5, freq="BME"):
        self.years = years
        self.freq = freq

        # Monthly returns over the full history (for a valid prior obs at the
        # window start).
        self.monthly_prices = prices.resample(freq).last()
        self.monthly_returns = self.monthly_prices.pct_change()

        self.benchmark_prices = benchmark.resample(freq).last()
        self.benchmark_returns = self.benchmark_prices.pct_change()


        # Shift signals forward one month so positions(t) line up with returns(t).
        positions = (
            signals.shift(1)
                   .reindex(index=self.monthly_returns.index,
                            columns=self.monthly_returns.columns)
        )
        self.positions = positions

        long_pos = positions.where(positions > 0, 0)
        short_pos = positions.where(positions < 0, 0)

        n_long = long_pos.abs().sum(axis=1).replace(0, pd.NA)
        n_short = short_pos.abs().sum(axis=1).replace(0, pd.NA)
        n_total = positions.abs().sum(axis=1).replace(0, pd.NA)

        # short_basket_returns is the *raw* avg return of the shorted names
        # (so it shows up positive when shorts went up against us)
        self.long_basket_returns = (self.monthly_returns * long_pos).sum(axis=1) / n_long
        self.short_basket_returns = (self.monthly_returns * (-short_pos)).sum(axis=1) / n_short
        self.portfolio_returns = (self.monthly_returns * positions).sum(axis=1) / n_total

        # Clip the report to the trailing window
        cutoff = self.monthly_returns.index.max() - pd.DateOffset(years=years)
        for attr in ("monthly_returns", "benchmark_returns",
                     "long_basket_returns", "short_basket_returns",
                     "portfolio_returns", "positions"):
            setattr(self, attr, getattr(self, attr).loc[cutoff:])

        self.cumulative_portfolio = (1 + self.portfolio_returns.fillna(0)).cumprod()
        self.cumulative_benchmark = (1 + self.benchmark_returns.fillna(0)).cumprod()
        self.cumulative_long = (1 + self.long_basket_returns.fillna(0)).cumprod()
        self.cumulative_short = (1 + self.short_basket_returns.fillna(0)).cumprod()

    def summary(self):
        """CAGR, annualized vol, Sharpe, max drawdown for each return series."""
        def stats(r):
            r = r.dropna()
            if r.empty:
                return pd.Series(dtype="float64")
            cum = (1 + r).cumprod()
            cagr = cum.iloc[-1] ** (12 / len(r)) - 1
            vol = r.std() * (12 ** 0.5)
            sharpe = (r.mean() * 12) / vol if vol else float("nan")
            dd = (cum / cum.cummax() - 1).min()
            return pd.Series({"CAGR": cagr, "AnnVol": vol,
                              "Sharpe": sharpe, "MaxDD": dd})
        return pd.DataFrame({
            "Portfolio": stats(self.portfolio_returns),
            "Long":      stats(self.long_basket_returns),
            "Short":     stats(self.short_basket_returns),
            "Benchmark": stats(self.benchmark_returns),
        })



In [ ]:
bt = MomentumBacktest(prices, signals, qqq, years=5)
bt.summary()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# monthly portfolio return as colored bars vs QQQ line.
fig, ax = plt.subplots(figsize=(12, 4))
port = bt.portfolio_returns
bar_colors = port.ge(0).map({True: "#2ca02c", False: "#d62728"})
ax.bar(port.index, port.values, width=20, color=bar_colors.values, label="L/S Portfolio")
ax.plot(bt.benchmark_returns.index, bt.benchmark_returns.values,
        color="black", linewidth=1.5, label="QQQ")
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_title("Monthly Portfolio Return vs QQQ")
ax.set_ylabel("Monthly Return")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.show()

# long picks vs short picks vs QQQ monthly returns.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(bt.long_basket_returns.index,  bt.long_basket_returns.values,
        label="Long basket",  color="#2ca02c", linewidth=1.5)
ax.plot(bt.short_basket_returns.index, bt.short_basket_returns.values,
        label="Short basket", color="#d62728", linewidth=1.5)
ax.plot(bt.benchmark_returns.index,    bt.benchmark_returns.values,
        label="QQQ",          color="black",   linewidth=1.5)
ax.axhline(0, color="gray", linewidth=0.5)
ax.set_title("Monthly Returns: Long vs Short vs QQQ")
ax.set_ylabel("Monthly Return")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend()
plt.tight_layout()
plt.show()

# cumulative growth of $1 - L/S portfolio vs QQQ.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(bt.cumulative_portfolio.index, bt.cumulative_portfolio.values,
        label="L/S Portfolio", linewidth=2)
ax.plot(bt.cumulative_benchmark.index, bt.cumulative_benchmark.values,
        label="QQQ", linewidth=2)
ax.set_title("Cumulative Return: L/S Portfolio vs QQQ")
ax.set_ylabel("Growth of $1")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# weekly returns heatmap

daily_returns = prices.pct_change()
sig_daily = signals.reindex(daily_returns.index, method="ffill").shift(1)

n_active = sig_daily.abs().sum(axis=1).replace(0, np.nan)
daily_port = (daily_returns * sig_daily).sum(axis=1) / n_active

# Clip to the same trailing window the backtest report uses, then compound
# daily P&L into Friday-ending weekly returns.
start = bt.portfolio_returns.index.min()
end = bt.portfolio_returns.index.max()
daily_port = daily_port.loc[start:end].fillna(0)
weekly_port = (1 + daily_port).resample("W-FRI").prod() - 1
weekly_port = weekly_port.loc[weekly_port.ne(0).cumsum() > 0]  # drop any leading empty weeks

# Pivot weekly returns into a Year x ISO-Week matrix for the heatmap.
iso = weekly_port.index.isocalendar()
heat = (
    pd.DataFrame({"Year": iso.year.values,
                  "Week": iso.week.values,
                  "Ret":  weekly_port.values})
      .pivot(index="Year", columns="Week", values="Ret")
      .reindex(columns=range(1, 54))
      .sort_index()
)

fig, ax = plt.subplots(figsize=(16, 0.6 * len(heat) + 2))
vmax = float(np.nanmax(np.abs(heat.values)))
im = ax.imshow(heat.values, aspect="auto", cmap="RdYlGn",
               vmin=-vmax, vmax=vmax, interpolation="nearest")

ax.set_xticks(range(0, 53, 2))
ax.set_xticklabels([str(w + 1) for w in range(0, 53, 2)])
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels([str(y) for y in heat.index])
ax.set_xlabel("ISO Week")
ax.set_ylabel("Year")
ax.set_title("Weekly L/S Portfolio Return Heatmap (Backtest Period)")

for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v * 100:.1f}", ha="center", va="center",
                    fontsize=6, color="black")

cbar = fig.colorbar(im, ax=ax, format=PercentFormatter(1.0))
cbar.set_label("Weekly Return")
plt.tight_layout()
plt.show()
